# CURE-Rec — run variations notebook

This notebook is the complete experiment control surface. Each run variation is isolated in its own commented cell so you can execute exactly the experiment you intend without editing the package code.

Run order:

1. run the end-to-end quick workflow once;
2. run the controlled regime suite;
3. calibrate/tune any failing regime;
4. run five-seed stabilization;
5. only then run 20-seed full experiments.

Do not activate all expensive cells at once.

## 1. Setup

Install once from `paper-ideas/CURE-Rec/code/`:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[dev]'
jupyter lab notebooks/00_cure_rec_quickstart.ipynb
```

In [ ]:
from pathlib import Path
import importlib
import json
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook from the CURE-Rec code directory or repository root.')

# VS Code kernels are long-lived: force local source and clear stale package modules.
sys.path[:] = [str(ROOT), *[entry for entry in sys.path if entry != str(ROOT)]]
for module_name in list(sys.modules):
    if module_name == 'cure_rec' or module_name.startswith('cure_rec.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

from cure_rec.analysis import analyze_dataset
from cure_rec.config import load_settings
from cure_rec.data import DatasetLoadResult, audit_interactions, load_dataset
from cure_rec.experiments import run_seed_sweep
from cure_rec.observability import RunLogger
from cure_rec.regimes import run_regime_suite
from cure_rec.workflow import run_full_workflow

import cure_rec.data as data_layer
assert hasattr(data_layer, 'DatasetLoadResult'), f'Stale data module loaded: {data_layer.__file__}'
print('Project root:', ROOT)
print('Data module:', data_layer.__file__)
print('CURE-Rec data layer: current')


## 2. Shared configuration

All variation cells use these locations. Data download is always explicit. MovieLens and Coat can be downloaded by the loader; Yahoo! R3 must be obtained manually under its access terms.

In [ ]:
DATA_ROOT = ROOT / 'data' / 'raw'
RUN_ROOT = ROOT / 'runs'

# Reuse these for public/local data variations.
MOVIELENS_SOURCE = DATA_ROOT / 'movielens_1m'
COAT_SOURCE = DATA_ROOT / 'coat'
YAHOO_SOURCE = DATA_ROOT / 'yahoo_r3'
LOCAL_CSV = None  # Set to Path('/path/to/interactions.csv') for the generic CSV variation.

def settings_for(mode: str, run_name: str):
    if mode not in {'quick', 'full'}:
        raise ValueError("mode must be 'quick' or 'full'")
    config_name = 'curesim_quickstart.yaml' if mode == 'quick' else 'curesim_full.yaml'
    settings = load_settings(ROOT / 'configs' / config_name)
    settings.run.name = run_name
    settings.run.output_root = RUN_ROOT
    return settings

print('Data root:', DATA_ROOT)
print('Run root:', RUN_ROOT)


## 3A. Data-only variation — MovieLens-1M

Fetch, standardize, audit, profile, and train registered CPU recommendation baselines. This is **not** the CURE-Sim causal run.

Set the switch to `True` only when you want to rerun external model analysis.

In [ ]:
RUN_ML1M_ANALYSIS = False

if RUN_ML1M_ANALYSIS:
    ml1m = load_dataset('movielens_1m', MOVIELENS_SOURCE, download=True)
    ml1m_audit = audit_interactions(ml1m.interactions)
    ml1m_analysis = analyze_dataset(
        ml1m,
        output_root=RUN_ROOT,
        run_bpr=True,
        bpr_updates=500_000,
        max_eval_users=1_000,
        seed=42,
    )
    print('Analysis run:', ml1m_analysis.run_dir)
    print('Evidence level:', ml1m_audit.permitted_claim)
    display(ml1m_analysis.summary)
    display(ml1m_analysis.model_metrics)
else:
    print('MovieLens analysis disabled.')


## 3B. Data-only variation — Coat

Coat is useful for randomized-vs-biased short-horizon estimator checks. It has no event timestamps or complete policy logs, so the audit should prevent long-horizon claims.

In [ ]:
RUN_COAT_ANALYSIS = False

if RUN_COAT_ANALYSIS:
    coat = load_dataset('coat', COAT_SOURCE, download=True)
    coat_audit = audit_interactions(coat.interactions)
    coat_analysis = analyze_dataset(coat, output_root=RUN_ROOT, run_bpr=False)
    print('Analysis run:', coat_analysis.run_dir)
    print('Evidence level:', coat_audit.permitted_claim)
    display(coat_analysis.summary)
else:
    print('Coat analysis disabled.')


## 3C. Data-only variation — Yahoo! R3 or a local CSV

Yahoo! R3 must already be available locally. The generic CSV variation is intentionally conservative and will report the strongest evidence level supported by its fields.

In [ ]:
RUN_YAHOO_ANALYSIS = False
RUN_LOCAL_CSV_ANALYSIS = False

if RUN_YAHOO_ANALYSIS:
    yahoo = load_dataset('yahoo_r3', YAHOO_SOURCE)
    yahoo_analysis = analyze_dataset(yahoo, output_root=RUN_ROOT, run_bpr=False)
    print('Yahoo! R3 analysis:', yahoo_analysis.run_dir)
    print('Evidence level:', yahoo_analysis.audit.permitted_claim)

if RUN_LOCAL_CSV_ANALYSIS:
    if LOCAL_CSV is None:
        raise ValueError('Set LOCAL_CSV before enabling RUN_LOCAL_CSV_ANALYSIS.')
    local = load_dataset('csv', LOCAL_CSV)
    local_analysis = analyze_dataset(local, output_root=RUN_ROOT, run_bpr=True, bpr_updates=500_000)
    print('Local CSV analysis:', local_analysis.run_dir)
    print('Evidence level:', local_analysis.audit.permitted_claim)

if not (RUN_YAHOO_ANALYSIS or RUN_LOCAL_CSV_ANALYSIS):
    print('Yahoo! R3 and local CSV variations disabled.')


## 4A. End-to-end quick variation — default first run

This is the recommended first execution. It fetches MovieLens if needed, audits and analyzes it, then runs the quick CURE-Sim causal workflow with all 64 coalitions, all quick scenarios, robust selection, and numbered assets.

In [ ]:
RUN_END_TO_END_QUICK = True

if RUN_END_TO_END_QUICK:
    quick_settings = settings_for('quick', 'curesim-quick-notebook')
    quick_workflow = run_full_workflow(
        quick_settings,
        dataset='movielens_1m',
        source=MOVIELENS_SOURCE,
        download=True,
        run_bpr=True,
        bpr_updates=500_000,
        max_eval_users=1_000,
    )
    ACTIVE_WORKFLOW = quick_workflow
    print('External-data analysis:', quick_workflow.analysis.run_dir)
    print('CURE run:', quick_workflow.cure_run_dir)
    print('Decision:', quick_workflow.decision.status.value, quick_workflow.decision.selected_interventions)
else:
    print('Quick end-to-end variation disabled.')


## 4B. End-to-end full variation — larger behavioural CURE-Sim

This variation is computationally expensive. It runs 120 users, 240 items, a 12-step horizon, four scenarios, exact 64-coalition evaluation, vectorized BPR-MF, and all assets. Enable only after the quick variation and regime suite are healthy.

In [ ]:
RUN_END_TO_END_FULL = False

if RUN_END_TO_END_FULL:
    full_settings = settings_for('full', 'curesim-full-notebook')
    full_workflow = run_full_workflow(
        full_settings,
        dataset='movielens_1m',
        source=MOVIELENS_SOURCE,
        download=True,
        run_bpr=True,
        bpr_updates=1_500_000,
        max_eval_users=1_000,
    )
    ACTIVE_WORKFLOW = full_workflow
    print('External-data analysis:', full_workflow.analysis.run_dir)
    print('CURE run:', full_workflow.cure_run_dir)
    print('Decision:', full_workflow.decision.status.value, full_workflow.decision.selected_interventions)
else:
    print('Full end-to-end variation disabled.')


## 5. Inspect the latest end-to-end workflow

This cell works after either quick or full end-to-end variation. It separates external-data baseline analysis from the causal CURE-Sim result.

In [ ]:
if 'ACTIVE_WORKFLOW' not in globals():
    print('Run variation 4A or 4B first.')
else:
    workflow = ACTIVE_WORKFLOW
    print('External-data evidence level:', workflow.analysis.audit.permitted_claim)
    display(workflow.analysis.summary)
    display(workflow.analysis.model_metrics)

    game = workflow.game
    RUN_DIR = workflow.cure_run_dir
    decision = workflow.decision
    display(game.regions.sort_values('phi_mean', ascending=False))
    display(game.interaction_table.sort_values('interaction_mean', ascending=False))

    coalitions = game.coalition_table.groupby('mask', as_index=False).agg(
        lower_improvement=('improvement', 'min'),
        upper_improvement=('improvement', 'max'),
        cost=('cost', 'first'),
        interventions=('active_interventions', 'first'),
    ).sort_values('lower_improvement', ascending=False)
    display(coalitions.head(12))


## 6. Controlled cooperative-structure regime suite

Run this before large behavioural seed sweeps. It verifies that the exact game, Shapley values, interaction values, and planner modes recover known additive, complementary, redundant, antagonistic, delayed, repair, and misspecified structures.

In [ ]:
RUN_REGIME_SUITE = False

if RUN_REGIME_SUITE:
    regime_settings = settings_for('quick', 'curesim-regime-suite')
    regime_logger = RunLogger(regime_settings)
    try:
        regime_suite = run_regime_suite(regime_settings, regime_logger)
        regime_logger.close(status='completed')
    except Exception:
        regime_logger.close(status='failed')
        raise
    print('Regime-suite assets:', regime_suite.run_dir)
    display(regime_suite.summary)
    display(regime_suite.attribution_recovery.groupby('regime', as_index=False).agg(
        shapley_mae=('absolute_error', 'mean'),
        sign_accuracy=('sign_correct', 'mean'),
    ))
else:
    print('Regime suite disabled.')


## 7A. Paired quick five-seed stabilization variation

This is the recommended first stochastic stability run after the regime suite passes. Common random numbers are shared within every seed across coalitions.

In [ ]:
RUN_QUICK_FIVE_SEEDS = False
QUICK_SEEDS = [42, 43, 44, 45, 46]

if RUN_QUICK_FIVE_SEEDS:
    quick_sweep_settings = settings_for('quick', 'curesim-quick-sweep')
    quick_sweep = run_seed_sweep(quick_sweep_settings, QUICK_SEEDS)
    print('Quick seed sweep:', quick_sweep.run_dir)
    display(quick_sweep.decisions)
    display(quick_sweep.attributions.groupby('intervention', as_index=False).agg(
        phi_mean=('phi_mean', 'mean'),
        phi_std=('phi_mean', 'std'),
        positive_rate=('phi_lower', lambda x: float((x > 0).mean())),
    ))
else:
    print('Quick five-seed sweep disabled.')


## 7B. Paired full five-seed stabilization variation

Run only after quick stabilization. This is the direct precursor to final multi-seed analysis and may take roughly an hour on the full configuration.

In [ ]:
RUN_FULL_FIVE_SEEDS = False
FULL_STABILIZATION_SEEDS = [42, 43, 44, 45, 46]

if RUN_FULL_FIVE_SEEDS:
    full_sweep_settings = settings_for('full', 'curesim-full-sweep')
    full_sweep = run_seed_sweep(full_sweep_settings, FULL_STABILIZATION_SEEDS)
    print('Full five-seed sweep:', full_sweep.run_dir)
    display(full_sweep.decisions)
    display(full_sweep.attributions.groupby('intervention', as_index=False).agg(
        phi_mean=('phi_mean', 'mean'),
        phi_std=('phi_mean', 'std'),
        positive_rate=('phi_lower', lambda x: float((x > 0).mean())),
    ))
else:
    print('Full five-seed sweep disabled.')


## 7C. Paired full 20-seed final variation

Do not run until all controlled regimes recover their expected structures and the five-seed full sweep is stable. This is the final statistical experiment variation and may take several hours.

In [ ]:
RUN_FULL_TWENTY_SEEDS = False
FINAL_SEEDS = list(range(100, 120))

if RUN_FULL_TWENTY_SEEDS:
    final_settings = settings_for('full', 'curesim-final-sweep')
    final_sweep = run_seed_sweep(final_settings, FINAL_SEEDS)
    print('Full 20-seed sweep:', final_sweep.run_dir)
    display(final_sweep.decisions)
    display(final_sweep.attributions.groupby('intervention', as_index=False).agg(
        phi_mean=('phi_mean', 'mean'),
        phi_std=('phi_mean', 'std'),
        positive_rate=('phi_lower', lambda x: float((x > 0).mean())),
    ))
else:
    print('Full 20-seed sweep disabled.')


## 8. Inspect all generated assets and logs

Run after variation 4A or 4B. The registry distinguishes generated CURE-Sim assets from manual literature work and future audited real-log OPE assets.

In [ ]:
if 'RUN_DIR' not in globals():
    print('Run quick or full end-to-end variation first.')
else:
    asset_manifest = json.loads((RUN_DIR / 'artifacts' / 'asset_manifest.json').read_text())
    display(pd.DataFrame(asset_manifest))

    print('Generated CURE tables:')
    for path in sorted((RUN_DIR / 'tables').glob('*.csv')):
        print('-', path.name)
    print('\nGenerated CURE figures:')
    for path in sorted((RUN_DIR / 'figures').glob('*.png')):
        print('-', path.name)

    events = pd.DataFrame([json.loads(line) for line in (RUN_DIR / 'logs' / 'events.jsonl').read_text().splitlines()])
    display(events[['timestamp_utc', 'event']].tail(20))
    display(json.loads((RUN_DIR / 'artifacts' / 'explanation_card.json').read_text()))


## 9. Scientific execution order

1. Run 4A once after code updates.
2. Run 6 and verify every controlled regime recovers its expected structure.
3. Run 7A for pipeline stabilization.
4. Diagnose any regime mismatch or baseline-feasibility anomaly.
5. Run 4B and 7B only after the previous steps pass.
6. Run 7C only for final multi-seed statistics.

External MovieLens analysis is useful for model/data robustness. The complete causal claims remain grounded in the controlled regime suite and CURE-Sim behavioural experiments.